In [3]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import tf_keras as keras
import os

# Carica features da disco
X_train = np.load("features/X_train.npy")
X_test = np.load("features/X_test.npy")
y_train = np.load("features/y_train.npy")
y_test = np.load("features/y_test.npy")
labels = np.load("features/labels.npy")

# ─────────────────────────────────────────────────────────────────────────
# Costruzione tf.data.Dataset pronti per il training
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,   # sklearn vuole numpy
    test_size=0.2,     # 20% validation
    stratify=y_train,  # mantiene distribuzione classi
    random_state=42
)

# Converti y in tensor
y_train_final = tf.convert_to_tensor(y_train_final)
y_val         = tf.convert_to_tensor(y_val)

# Converti X in float32 (range approssimativo [-1, 1] dopo rescaling)
# Il modello riceve (49, 32, 1) – aggiunta dim canale per Conv2D.
def make_dataset(X: np.ndarray, y: tf.Tensor, shuffle: bool) -> tf.data.Dataset:
    # float32 normalizzato in [-1, 1] per facilitare il training
    X_f = (X.astype(np.float32) / 128.0).reshape(len(X), -1)  # (N, 1568)
    ds  = tf.data.Dataset.from_tensor_slices((X_f, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_final, y_train_final, shuffle=True)
val_ds   = make_dataset(X_val,   y_val,   shuffle=False)
test_ds  = make_dataset(X_test,  y_test,  shuffle=False)

print("\ntrain_ds:", train_ds)
print("val_ds  :", val_ds)
print("test_ds :", test_ds)


train_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>
val_ds  : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>
test_ds : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>


In [ ]:
import tensorflow as tf

EPOCHS = 100
BATCH_SIZE = 32
num_classes = 5

# Test MLP
model = keras.Sequential([
    keras.layers.Input(shape=(1568,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                50208     
                                                                 
 dense_1 (Dense)             (None, 16)                528       
                                                                 
 dense_2 (Dense)             (None, 8)                 136       
                                                                 
 dense_3 (Dense)             (None, 5)                 45        
                                                                 
Total params: 50917 (198.89 KB)
Trainable params: 50917 (198.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [3]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/100
66/66 [==============================] - 1s 4ms/step - loss: 1.3727 - accuracy: 0.4527 - val_loss: 1.2010 - val_accuracy: 0.5335
Epoch 2/100
66/66 [==============================] - 0s 2ms/step - loss: 1.0755 - accuracy: 0.5865 - val_loss: 1.0034 - val_accuracy: 0.6291
Epoch 3/100
66/66 [==============================] - 0s 2ms/step - loss: 0.8722 - accuracy: 0.6697 - val_loss: 0.8958 - val_accuracy: 0.6348
Epoch 4/100
66/66 [==============================] - 0s 2ms/step - loss: 0.7283 - accuracy: 0.7251 - val_loss: 0.7971 - val_accuracy: 0.6883
Epoch 5/100
66/66 [==============================] - 0s 2ms/step - loss: 0.6141 - accuracy: 0.7811 - val_loss: 0.7780 - val_accuracy: 0.6960
Epoch 6/100
66/66 [==============================] - 0s 2ms/step - loss: 0.5253 - accuracy: 0.8112 - val_loss: 0.7484 - val_accuracy: 0.7247
Epoch 7/100
66/66 [==============================] - 0s 2ms/step - loss: 0.4573 - accuracy: 0.8365 - val_loss: 0.7610 - val_accuracy: 0.7323
Epoch 8/100
6

In [4]:
# Valutazione
test_loss, test_acc = model.evaluate(test_ds)

print("Test accuracy:", test_acc)

21/21 [==============================] - 0s 1ms/step - loss: 1.7956 - accuracy: 0.7114
Test accuracy: 0.7113884687423706


In [5]:
# Only to test QAT
model.save_weights("models/QAware_MLP_weights.h5")

In [6]:
# Quantization Aware Training (QAT)
import tensorflow_model_optimization as tfmot
import tf_keras as keras

EPOCHS = 10

model_QAware = keras.Sequential([
    keras.layers.Input(shape=(1568,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

model_QAware.load_weights("models/QAware_MLP_weights.h5")

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model_QAware)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=keras.losses.CategoricalCrossentropy(),
              metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 1568)              3         
 yer)                                                            
                                                                 
 quant_dense_4 (QuantizeWra  (None, 32)                50213     
 pperV2)                                                         
                                                                 
 quant_dense_5 (QuantizeWra  (None, 16)                533       
 pperV2)                                                         
                                                                 
 quant_dense_6 (QuantizeWra  (None, 8)                 141       
 pperV2)                                                         
                                                                 
 quant_dense_7 (QuantizeWra  (None, 5)                

In [7]:
q_aware_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

Epoch 1/10
66/66 [==============================] - 1s 5ms/step - loss: 0.2551 - accuracy: 0.9197 - val_loss: 1.0305 - val_accuracy: 0.6826
Epoch 2/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0900 - accuracy: 0.9828 - val_loss: 1.0670 - val_accuracy: 0.7170
Epoch 3/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0457 - accuracy: 0.9914 - val_loss: 1.1392 - val_accuracy: 0.7400
Epoch 4/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0271 - accuracy: 0.9957 - val_loss: 1.1912 - val_accuracy: 0.7438
Epoch 5/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0183 - accuracy: 0.9957 - val_loss: 1.2415 - val_accuracy: 0.7304
Epoch 6/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0161 - accuracy: 0.9957 - val_loss: 1.2796 - val_accuracy: 0.7323
Epoch 7/10
66/66 [==============================] - 0s 3ms/step - loss: 0.0183 - accuracy: 0.9957 - val_loss: 1.3619 - val_accuracy: 0.7476
Epoch 8/10
66/66 [==

In [8]:
_, q_aware_model_accuracy = q_aware_model.evaluate(test_ds)

print('Quant test accuracy:', q_aware_model_accuracy)

21/21 [==============================] - 0s 2ms/step - loss: 1.5452 - accuracy: 0.7348
Quant test accuracy: 0.7347893714904785


In [5]:
# Post training quantization (PTQ)
model = keras.Sequential([
    keras.layers.Input(shape=(1568,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(5, activation='softmax')
])

model.load_weights("models/QAware_MLP_weights.h5")

In [6]:
# Convert to TFLite full int8
def representative_dataset():
    idx = np.random.choice(len(X_train), 300, replace=False)
    for i in idx:
        data = X_train[i].astype(np.float32)
        data = np.expand_dims(data, axis=0)
        yield [data]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

save_path = "models/MLP.tflite"

with open(save_path, "wb") as f:
    f.write(tflite_quant_model)

print("Modello salvato in:", save_path)

INFO:tensorflow:Assets written to: /tmp/tmp8zcdqa1a/assets


INFO:tensorflow:Assets written to: /tmp/tmp8zcdqa1a/assets
/home/nicola/KeywordSpotting/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1776510640.373533    4432 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776510640.373548    4432 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-04-18 13:10:40.375518: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp8zcdqa1a
2026-04-18 13:10:40.376107: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-04-18 13:10:40.376113: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp8zcdqa1a
I0000 00:00:1776510640.379485    4432 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-04-18 13:10:40.380012: I tensorflow/cc/saved

Modello salvato in: models/MLP.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
2026-04-18 13:10:40.662111: W tensorflow/compiler/mlir/lite/flatbuffer_export.cc:3705] Skipping runtime version metadata in the model. This will be generated by the exporter.
